# 05 · Project Scope Service

The **Scope Project** service transforms a client's free-form project description into a
structured, professional scope document, enriched with competitive intelligence and brand direction.

## Steps

1. **Contradiction check** — LLM scans the brief for internal inconsistencies and asks clarifying questions if found.
2. **Competitor scraping** — Tavily searches and scrapes up to 3 competitor sites to inform the scope.
3. **Scope synthesis** — LLM produces a structured JSON scope document (objectives, features, timeline, risks).
4. **Brand palette generation** — LLM generates a 5-colour brand palette with hex codes and rationale.

## Architecture

```
Client brief (text)
      │
      ▼
[Contradiction check]
      │
      ├─ contradictions? ──► return clarification message
      │
      ▼
[Competitor discovery via Tavily]
      │
      ▼
[Competitor scraping via httpx]
      │
      ▼
[Scope synthesis + Palette generation]
      │
      ▼
Structured scope JSON + colour palette
```

In [1]:
%pip install -q langchain-groq tavily-python httpx beautifulsoup4 python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import httpx
from bs4 import BeautifulSoup
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from tavily import TavilyClient

load_dotenv(dotenv_path=Path('..') / '.env')

llm    = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.4)
tavily = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

print('Ready')

Ready


## Sample Project Brief

We use a realistic brief for a fictional wellness app.

In [3]:
sample_brief = {
    'company_name': 'ZenFlow',
    'industry': 'Wellness & Mental Health',
    'description': (
        'ZenFlow is a mindfulness and meditation app targeting busy professionals aged 28–45. '
        'We want an app that helps users meditate for 10–30 minutes daily, tracks their mood, '
        'integrates with wearables (Apple Watch, Fitbit), and provides AI-personalised meditation paths. '
        'We need this built in 6 weeks with a team of 2 developers and want it to be free with no revenue model yet.'
    ),
    'timeline': '6 weeks',
    'budget': '$5,000',
    'competitors': ['Calm', 'Headspace', 'Insight Timer'],
    'target_audience': 'Professionals aged 28–45',
    'primary_goals': ['reduce user stress', 'build daily meditation habit', 'track mood over time'],
}

brief_text = f"""
Company: {sample_brief['company_name']}
Industry: {sample_brief['industry']}

Description:
{sample_brief['description']}

Timeline: {sample_brief['timeline']}
Budget: {sample_brief['budget']}
Target Audience: {sample_brief['target_audience']}
Primary Goals: {', '.join(sample_brief['primary_goals'])}
""".strip()

print(brief_text)

Company: ZenFlow
Industry: Wellness & Mental Health

Description:
ZenFlow is a mindfulness and meditation app targeting busy professionals aged 28–45. We want an app that helps users meditate for 10–30 minutes daily, tracks their mood, integrates with wearables (Apple Watch, Fitbit), and provides AI-personalised meditation paths. We need this built in 6 weeks with a team of 2 developers and want it to be free with no revenue model yet.

Timeline: 6 weeks
Budget: $5,000
Target Audience: Professionals aged 28–45
Primary Goals: reduce user stress, build daily meditation habit, track mood over time


## Step 1 — Contradiction Check

In [4]:
CONTRADICTION_PROMPT = """
You are a product strategist. Review the project brief and identify any internal contradictions,
unrealistic expectations, or missing critical information.

Common issues to check:
- Timeline too short for the scope
- Budget insufficient for stated features  
- Goals that conflict with each other
- Missing technical or business details

Respond with JSON: {"has_contradictions": true/false, "issues": ["..."], "clarifications": ["..."]}
If no issues, return: {"has_contradictions": false, "issues": [], "clarifications": []}
""".strip()

async def check_contradictions(brief: str) -> dict:
    msg = await llm.ainvoke([
        SystemMessage(content=CONTRADICTION_PROMPT),
        HumanMessage(content=brief),
    ])
    try:
        raw = msg.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1].lstrip('json').strip()
        return json.loads(raw)
    except Exception:
        return {'has_contradictions': False, 'issues': [], 'clarifications': []}

contradictions = await check_contradictions(brief_text)
print('Has contradictions:', contradictions['has_contradictions'])
if contradictions['issues']:
    print('\nIssues found:')
    for issue in contradictions['issues']:
        print(f'  ⚠ {issue}')
if contradictions['clarifications']:
    print('\nClarifications needed:')
    for q in contradictions['clarifications']:
        print(f'  ? {q}')

Has contradictions: True

Issues found:
  ⚠ Timeline too short for the scope: 6 weeks is an extremely aggressive timeline for building a complex app with wearable integrations and AI-personalised meditation paths.
  ⚠ Budget insufficient for stated features: $5,000 is a very low budget for developing a high-quality app with multiple features, integrations, and AI capabilities.
  ⚠ Lack of revenue model: launching an app with no revenue model may not be sustainable in the long term.

Clarifications needed:
  ? What is the expected user acquisition strategy and marketing budget?
  ? Are there any existing technical infrastructure or resources that can be leveraged to support the development of the app?
  ? What is the planned approach for developing and training the AI-personalised meditation paths?
  ? Are there any specific wearable integration requirements or limitations that need to be considered?


## Step 2 — Competitor Discovery (Tavily)

In [5]:
def discover_competitors(brief: str, industry: str) -> list:
    """Use Tavily to find competitor URLs."""
    query = f'top competitors {industry} similar to: {brief[:200]}'
    try:
        result = tavily.search(query=query, max_results=5, search_depth='basic')
        urls = [r['url'] for r in result.get('results', []) if r.get('url')]
        # Filter out non-competitor URLs
        excluded = ('reddit', 'wikipedia', 'youtube', 'twitter', 'facebook')
        filtered = [u for u in urls if not any(kw in u.lower() for kw in excluded)]
        return filtered[:3]
    except Exception as e:
        print(f'Tavily search failed: {e}')
        return []

competitor_urls = discover_competitors(brief_text, sample_brief['industry'])
print('Found competitor URLs:')
for url in competitor_urls:
    print(f'  • {url}')

Found competitor URLs:
  • https://soulsyncblog.com/best-wellness-apps/mindfulness-apps
  • https://www.cnet.com/health/mental/best-mental-health-apps/
  • https://www.verywellmind.com/best-meditation-apps-4767322


## Step 3 — Scope Synthesis

In [6]:
SCOPE_PROMPT = """
You are a senior product strategist at a digital design studio.
Based on the project brief, produce a professional project scope document in JSON format with these keys:

{
  "project_title": "...",
  "executive_summary": "...",
  "objectives": ["..."],
  "core_features": [{"feature": "...", "priority": "must-have|nice-to-have", "effort": "low|medium|high"}],
  "out_of_scope": ["..."],
  "technical_requirements": ["..."],
  "suggested_timeline": {"phases": [{"name": "...", "duration": "...", "deliverables": ["..."]}]},
  "risks": [{"risk": "...", "mitigation": "..."}],
  "success_metrics": ["..."]
}

Be specific and realistic. Flag any scope items that exceed the stated budget/timeline.
""".strip()

async def synthesize_scope(brief: str, competitor_context: str = '') -> dict:
    content = brief
    if competitor_context:
        content += f'\n\n## Competitor Landscape\n{competitor_context}'

    msg = await llm.ainvoke([
        SystemMessage(content=SCOPE_PROMPT),
        HumanMessage(content=content),
    ])
    try:
        raw = msg.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1].lstrip('json').strip()
        return json.loads(raw)
    except Exception:
        return {'raw_scope': msg.content}

scope = await synthesize_scope(brief_text)
print('Scope keys:', list(scope.keys()))
print('\nProject title:', scope.get('project_title', 'N/A'))
print('\nObjectives:')
for obj in scope.get('objectives', []):
    print(f'  • {obj}')
print('\nCore features:')
for feat in scope.get('core_features', [])[:5]:
    print(f'  [{feat.get("priority", "?")}] {feat.get("feature", "?")}')

Scope keys: ['project_title', 'executive_summary', 'objectives', 'core_features', 'out_of_scope', 'technical_requirements', 'suggested_timeline', 'risks', 'success_metrics']

Project title: ZenFlow Mindfulness App

Objectives:
  • Design and develop a user-friendly meditation app for busy professionals
  • Implement mood tracking and wearable integration features
  • Develop AI-personalised meditation paths to enhance user experience
  • Launch the app within 6 weeks with a team of 2 developers

Core features:
  [must-have] Meditation sessions with guided audio
  [must-have] Mood tracking with user input and emotional analytics
  [nice-to-have] Integration with Apple Watch and Fitbit wearables
  [nice-to-have] AI-personalised meditation paths based on user data
  [must-have] User profile and meditation history


## Step 4 — Brand Palette Generation

In [7]:
PALETTE_PROMPT = """
You are a brand designer. Based on the project brief, generate a brand colour palette with exactly 5 colours.

For each colour provide:
- name (descriptive)
- hex code (valid CSS hex, e.g. #4A2C7F)
- role (primary / secondary / accent / background / text)
- rationale (why this colour for this brand)

Consider the industry, target audience, and emotional tone of the brand.
Respond with JSON: {"palette": [{"name": "...", "hex": "...", "role": "...", "rationale": "..."}]}
""".strip()

async def generate_palette(brief: str) -> list:
    msg = await llm.ainvoke([
        SystemMessage(content=PALETTE_PROMPT),
        HumanMessage(content=brief),
    ])
    try:
        raw = msg.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1].lstrip('json').strip()
        data = json.loads(raw)
        return data.get('palette', [])
    except Exception:
        return []

palette = await generate_palette(brief_text)
print('Generated palette:')
for swatch in palette:
    print(f'  {swatch["hex"]}  [{swatch["role"]:12s}]  {swatch["name"]}')
    print(f'           {swatch.get("rationale", "")[:80]}')

Generated palette:
  #4567b7  [primary     ]  Mindful Blue
           This calming blue tone evokes feelings of serenity and tranquility, perfect for 
  #f7f7f7  [background  ]  Calm Cream
           A soft, neutral cream color that provides a clean and calming background for the
  #8bc34a  [accent      ]  Nature's Green
           This muted green tone is reminiscent of nature and growth, symbolizing the posit
  #6495ed  [secondary   ]  Deep Breath
           A soothing blue-green color that represents the idea of taking a deep breath and
  #333333  [text        ]  Textured Grey
           A dark, neutral grey color that provides sufficient contrast with the background


## Summary

- Contradiction detection catches misaligned client expectations **before** design work starts.
- Competitor discovery expands the brief with real market context automatically.
- Palette generation ensures the scope document includes actionable brand direction from day 1.
- The entire pipeline runs in a single API call from the client's perspective.